# Lesson 07: KPI Dashboards with Plotly + Oracle

**Goal:** Query Oracle for KPIs, load into pandas, visualize with Plotly Express.

**Prerequisites:** Run `01_enrich_schema.sql` and `02_seed_dashboard_data.sql` in FreeSQL first.

## Step 1: Install Dependencies

In [1]:
!pip install -q oracledb sqlalchemy pandas plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.1 MB/s eta 0:00:00


## Step 2: Connect to Oracle FreeSQL

In [4]:
import oracledb
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine

# Replace with your FreeSQL credentials
USERNAME = ""
PASSWORD = ""  # <-- Replace!
DSN = "tcps://db.freesql.com:2484/23ai_34ui2"

# Create SQLAlchemy engine (used by pandas read_sql)
engine = create_engine(
    "oracle+oracledb://:@",
    connect_args={
        "user": USERNAME,
        "password": PASSWORD,
        "dsn": DSN
    }
)

# Quick test
df_test = pd.read_sql("SELECT COUNT(*) AS task_count FROM tasks", engine)
print(f"Connected! Tasks in database: {df_test.iloc[0]['task_count']}")

Connected! Tasks in database: 30


## Step 3: KPI 1 — Tasks by Status (Pie Chart)

In [5]:
query = """
SELECT status,
       COUNT(*) AS task_count,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
FROM   tasks
GROUP  BY status
ORDER  BY task_count DESC
"""

df_status = pd.read_sql(query, engine)

fig = px.pie(
    df_status,
    values='task_count',
    names='status',
    title='Tasks by Status',
    hole=0.4,  # Donut chart
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## Step 4: KPI 2 — Tasks per Team (Bar Chart)

In [6]:
query = """
SELECT t.name AS team_name,
       COUNT(ts.id) AS task_count
FROM   teams t
LEFT   JOIN users u ON u.team_id = t.id
LEFT   JOIN tasks ts ON ts.assigned_to = u.id
GROUP  BY t.id, t.name
ORDER  BY task_count DESC
"""

df_team = pd.read_sql(query, engine)

fig = px.bar(
    df_team,
    x='team_name',
    y='task_count',
    title='Tasks per Team',
    color='team_name',
    text='task_count'
)
fig.update_layout(showlegend=False)
fig.show()

## Step 5: KPI 3 — Workload per User (Horizontal Bar)

In [7]:
query = """
SELECT u.full_name,
       t.name AS team_name,
       COUNT(ts.id) AS open_tasks
FROM   users u
JOIN   teams t ON t.id = u.team_id
LEFT   JOIN tasks ts ON ts.assigned_to = u.id
                    AND ts.status IN ('open', 'in_progress', 'blocked')
GROUP  BY u.id, u.full_name, t.name
ORDER  BY open_tasks DESC
"""

df_workload = pd.read_sql(query, engine)

fig = px.bar(
    df_workload,
    x='open_tasks',
    y='full_name',
    color='team_name',
    title='Current Workload per User (Open + In Progress + Blocked)',
    orientation='h',
    text='open_tasks'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## Step 6: KPI 4 & 5 — Completion Rate + Avg Resolution (KPI Cards)

In [8]:
query = """
SELECT ROUND(
           COUNT(CASE WHEN status = 'completed' THEN 1 END) * 100.0
           / NULLIF(COUNT(CASE WHEN status != 'cancelled' THEN 1 END), 0),
           1
       ) AS completion_rate_pct,
       ROUND(AVG(
           EXTRACT(DAY FROM (completed_at - created_at)) * 24 +
           EXTRACT(HOUR FROM (completed_at - created_at)) +
           EXTRACT(MINUTE FROM (completed_at - created_at)) / 60
       ), 1) AS avg_resolution_hours
FROM   tasks
WHERE  status = 'completed'
  AND  completed_at IS NOT NULL
"""

df_kpi = pd.read_sql(query, engine)

completion_rate = df_kpi.iloc[0]['completion_rate_pct']
avg_hours = df_kpi.iloc[0]['avg_resolution_hours']

# Display as styled KPI cards
from IPython.display import display, HTML

html = f"""
<div style='display: flex; gap: 20px;'>
  <div style='background: #e8f5e9; padding: 20px; border-radius: 10px; min-width: 150px; text-align: center;'>
    <div style='font-size: 32px; font-weight: bold; color: #2e7d32;'>{completion_rate}%</div>
    <div style='color: #555;'>Completion Rate</div>
  </div>
  <div style='background: #e3f2fd; padding: 20px; border-radius: 10px; min-width: 150px; text-align: center;'>
    <div style='font-size: 32px; font-weight: bold; color: #1565c0;'>{avg_hours}h</div>
    <div style='color: #555;'>Avg Resolution</div>
  </div>
</div>
"""

display(HTML(html))

## Step 7: KPI 6 — Tasks Created per Day (Line Chart)

In [ ]:
query = """
SELECT TRUNC(created_at) AS creation_date,
       COUNT(*) AS tasks_created
FROM   tasks
GROUP  BY TRUNC(created_at)
ORDER  BY creation_date
"""

df_daily = pd.read_sql(query, engine)
df_daily['creation_date'] = pd.to_datetime(df_daily['creation_date'])

fig = px.line(
    df_daily,
    x='creation_date',
    y='tasks_created',
    title='Tasks Created per Day',
    markers=True
)
fig.update_xaxes(title_text='Date')
fig.update_yaxes(title_text='Tasks Created', dtick=1)
fig.show()

## Step 8: KPI 7 — Overdue Tasks (Alert Table)

In [9]:
query = """
SELECT ts.title,
       ts.status,
       ts.priority,
       ts.due_date,
       u.full_name AS assignee,
       t.name AS team
FROM   tasks ts
LEFT   JOIN users u ON u.id = ts.assigned_to
LEFT   JOIN teams t ON t.id = u.team_id
WHERE  ts.due_date < TRUNC(SYSDATE)
  AND  ts.status NOT IN ('completed', 'cancelled')
  AND  ts.due_date IS NOT NULL
ORDER  BY ts.due_date
"""

df_overdue = pd.read_sql(query, engine)

# Color-code by priority
priority_colors = {'critical': '#d32f2f', 'high': '#f57c00', 'medium': '#fbc02d', 'low': '#388e3c'}

fig = px.bar(
    df_overdue,
    x='title',
    y='due_date',
    color='priority',
    color_discrete_map=priority_colors,
    title=f'Overdue Tasks ({len(df_overdue)} total)',
    hover_data=['assignee', 'team', 'status']
)
fig.update_xaxes(title_text='Task')
fig.update_yaxes(title_text='Due Date')
fig.show()

# Also show as a clean table
print("\nOverdue Tasks Detail:")
print(df_overdue.to_string(index=False))


Overdue Tasks Detail:
                     title      status priority   due_date    assignee        team
     Mobile responsive nav     blocked     high 2026-05-07 Carol White     Product
         API rate limiting        open     high 2026-05-08 Alice Smith Engineering
           Two-factor auth        open critical 2026-05-09   Bob Jones Engineering
           Fix memory leak     blocked critical 2026-05-09 Alice Smith Engineering
 Write unit tests for auth in_progress   medium 2026-05-09   Bob Jones Engineering
      Design new dashboard in_progress   medium 2026-05-10 Carol White     Product
       Redis caching layer        open     high 2026-05-10 Alice Smith Engineering
Error tracking integration        open   medium 2026-05-11 Alice Smith Engineering
       Session timeout bug        open     high 2026-05-11   Bob Jones Engineering
          GDPR data export        open     high 2026-05-12 Alice Smith Engineering
     Set up CI/CD pipeline in_progress   medium 2026-05-12   Bob

## Step 9: KPI 8 — Priority Distribution (Stacked Bar)

In [ ]:
query = """
SELECT priority,
       COUNT(CASE WHEN status = 'open'        THEN 1 END) AS open_count,
       COUNT(CASE WHEN status = 'in_progress' THEN 1 END) AS in_progress_count,
       COUNT(CASE WHEN status = 'blocked'      THEN 1 END) AS blocked_count,
       COUNT(CASE WHEN status = 'completed'   THEN 1 END) AS completed_count
FROM   tasks
GROUP  BY priority
ORDER  BY CASE priority
              WHEN 'critical' THEN 1
              WHEN 'high'     THEN 2
              WHEN 'medium'   THEN 3
              WHEN 'low'      THEN 4
          END
"""

df_priority = pd.read_sql(query, engine)

# Melt for stacked bar format
df_melted = df_priority.melt(
    id_vars=['priority'],
    value_vars=['open_count', 'in_progress_count', 'blocked_count', 'completed_count'],
    var_name='status',
    value_name='count'
)

# Clean up status labels
status_map = {
    'open_count': 'open',
    'in_progress_count': 'in_progress',
    'blocked_count': 'blocked',
    'completed_count': 'completed'
}
df_melted['status'] = df_melted['status'].map(status_map)

fig = px.bar(
    df_melted,
    x='priority',
    y='count',
    color='status',
    title='Priority Distribution by Status',
    barmode='stack'
)
fig.show()

## Step 10: Compose the Full Dashboard

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create a 2x2 grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Tasks by Status',
        'Tasks per Team',
        'Tasks Created per Day',
        'Workload per User'
    ),
    specs=[
        [{'type': 'pie'}, {'type': 'bar'}],
        [{'type': 'scatter'}, {'type': 'bar'}]
    ]
)

# 1. Pie chart (status)
fig.add_trace(
    go.Pie(
        labels=df_status['status'],
        values=df_status['task_count'],
        hole=0.4,
        name='Status'
    ),
    row=1, col=1
)

# 2. Bar chart (team)
fig.add_trace(
    go.Bar(
        x=df_team['team_name'],
        y=df_team['task_count'],
        name='Team',
        marker_color='steelblue'
    ),
    row=1, col=2
)

# 3. Line chart (daily)
fig.add_trace(
    go.Scatter(
        x=df_daily['creation_date'],
        y=df_daily['tasks_created'],
        mode='lines+markers',
        name='Daily',
        line=dict(color='green')
    ),
    row=2, col=1
)

# 4. Horizontal bar (workload)
fig.add_trace(
    go.Bar(
        x=df_workload['open_tasks'],
        y=df_workload['full_name'],
        name='Workload',
        orientation='h',
        marker_color='coral'
    ),
    row=2, col=2
)

fig.update_layout(
    title_text='Task Management Dashboard',
    height=700,
    showlegend=False
)

fig.show()

## What You Learned

1. **SQL computes, Python displays** — The database does aggregation, filtering, date math. Python only presents.
2. **`pd.read_sql()` is the bridge** — One line connects Oracle to pandas.
3. **Plotly Express is fast** — ~5 lines per chart type. Interactive by default.
4. **KPIs need definitions** — Every metric has edge cases. Handle them in SQL.

## Exercise

Add a new chart: **Tasks completed per day** (line chart).
Hint: Filter `WHERE status = 'completed'` and group by `TRUNC(completed_at)`.